<div style='text-align:center;'>
<figure><img src='https://raw.githubusercontent.com/wekeo/wekeo4data/main/img/LogoWekeo_Copernicus_RGB_0.png' alt='Logo EU Copernicus WEkEO' align='right' width='20%'>
</figure>
</div>

<h1><center><code>How To Download WEkEO Data</code></center></h1>

Follow the next few steps to download data from WEkEO via the __HDA API__.  
Please check the following article to get further details: 
- [How to use the HDA API in Python?
](https://help.wekeo.eu/en/articles/6751608-how-to-use-the-hda-api-in-python)
- [How to download WEkEO data?](https://help.wekeo.eu/en/articles/6416936-how-to-download-wekeo-data)
- [Official documentation of HDA API](https://hda.readthedocs.io/en/latest/usage.html)

## Step 1. Install the latest version of `hda` and import module

In [2]:
!pip install hda -U

  Using cached hda-2.39-py3-none-any.whl.metadata (13 kB)
  Using cached tqdm-4.67.3-py3-none-any.whl.metadata (57 kB)
Using cached hda-2.39-py3-none-any.whl (22 kB)
Using cached tqdm-4.67.3-py3-none-any.whl (78 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [hda]


In [1]:
from hda import Client, Configuration

## Step 2. Configure credentials and load `hda` Client

In [2]:
# Configure your credentials without a .hdarc file
conf = Configuration(user = "gkodl", password = "Europe@4space")  # add your Wekeo username and password here
hda_client = Client(config = conf)

## Step 3. Create the request and download data

### Get the dataset metadata

Here we are going to download the following Copernicus Land dataset: __EO:EEA:DAT:CLMS_HRVPP_VPP__.

To create our request we can ask to the API what parameters are needed.
To do so we use the `metadata()` function:

In [3]:
# Request metadata of a dataset
hda_client.metadata(dataset_id="EO:EEA:DAT:CLMS_HRVPP_VPP")

{'type': 'object',
 'title': 'Queryable',
 'properties': {'dataset_id': {'title': 'dataset_id',
   'type': 'string',
   'oneOf': [{'const': 'EO:EEA:DAT:CLMS_HRVPP_VPP',
     'title': 'EO:EEA:DAT:CLMS_HRVPP_VPP',
     'group': None}]},
  'itemsPerPage': {'title': 'Items PerPage',
   'type': 'string',
   'pattern': '^[0-9]*$'},
  'startIndex': {'title': 'Start Index',
   'type': 'string',
   'pattern': '^[0-9]*$'},
  'uid': {'title': 'Uid', 'type': 'string', 'pattern': '[\\w-]+'},
  'productType': {'title': 'Product Type',
   'type': 'string',
   'oneOf': [{'const': 'MINV', 'title': 'MINV', 'group': None},
    {'const': 'MAXD', 'title': 'MAXD', 'group': None},
    {'const': 'LENGTH', 'title': 'LENGTH', 'group': None},
    {'const': 'SOSD', 'title': 'SOSD', 'group': None},
    {'const': 'QFLAG', 'title': 'QFLAG', 'group': None},
    {'const': 'EOSV', 'title': 'EOSV', 'group': None},
    {'const': 'TPROD', 'title': 'TPROD', 'group': None},
    {'const': 'MAXV', 'title': 'MAXV', 'group': No

## Create the request

Based on this information we can create the request below.

<div class="alert alert-block alert-info">
    📌 <b>Note</b>: to learn how to get your query from the Data Viewer, please check <a href="https://help.wekeo.eu/en/articles/6416936-how-to-download-wekeo-data#h_85849dcd7a">this article</a>.
</div>

## Datasets  configuration

Download following CLMS datasets:
1) CLCplus Backbone
2) Bare Soil Before
3) Bare Soil After
4) Small Woody Features
5) Grassland
6) Grassland Mowing Events
7) Tree cover density

In [25]:
LAYER_TEMPLATES = {
    "clcplus_backbone": {
        "dataset_id": "EO:EEA:DAT:CLC-PLUS",
        "productType": "Raster Layer",
        "resolution": "10m",
    },
    "bare_soil_before": {
        "dataset_id": "EO:EEA:DAT:HRL:CRL",
        "productType": "Bare Soil Before",
        "resolution": "10m",
    },
    "bare_soil_after": {
        "dataset_id": "EO:EEA:DAT:HRL:CRL",
        "productType": "Bare Soil After",
        "resolution": "10m",
    },
    "small_woody_features": {
        "dataset_id": "EO:EEA:DAT:HRL:SLF",
        "productType": "Small Woody Features",
        "resolution": "5m",
    },
    "grassland": {
        "dataset_id": "EO:EEA:DAT:HRL:GRA",
        "productType": "Grassland",
        "resolution": "10m",
    },
    "grassland_mowing_events": {
        "dataset_id": "EO:EEA:DAT:HRL:GRA",
        "productType": "Grassland Mowing Events",
        "resolution": "10m",
    },
    "tree_cover_density": {
        "dataset_id": "EO:EEA:DAT:HRL:TCF",
        "productType": "Tree Cover Density",
        "resolution": "10m",  
    },
}

In [26]:
# Bounding boxes dictionary
BBOXES = {
    'dnk': (8.076389, 54.559029, 15.193056, 57.751526),      # Denmark
    'nld': (3.360782, 50.723492, 7.227095, 53.554585),       # Netherlands
    'uk': (-4.723103, 50.201444, 1.764393, 52.998411),       # UK
}

def download_layer(layer_key, country, year):
    template = LAYER_TEMPLATES[layer_key]

    min_lon, min_lat, max_lon, max_lat = BBOXES[country]

    # Build final query
    query = {
        "dataset_id": template["dataset_id"],
        "productType": template["productType"],
        "resolution": template["resolution"],
        "year": str(year),
        "bbox": [min_lon, min_lat, max_lon, max_lat],
        "itemsPerPage": 200,
        "startIndex": 0,
    }

    print(f"\n=== {layer_key} | {country.upper()} | {year} ===")
    print(query)

    matches = hda_client.search(query)
    print(f"Found {len(matches)} items")

    out_path = f"../../data/LEON_P5_BII/EO_data_raw/{layer_key}/{layer_key}_{country}_{year}"
    matches.download(out_path)

    print(f"✓ Saved to {out_path}")

## Download Data

In [24]:
countries = ["dnk", "nld"]
years = range(2018, 2024)

layers_to_download = [
    "grassland_mowing_events",  # can be added "clcplus_backbone", "bare_soil_before", "bare_soil_after", "small_woody_features", "grassland", "grassland_mowing_events", "tree_cover_density"
]

for layer in layers_to_download:
    for country in countries:
        for year in years:
            download_layer(layer, country, year)


=== grassland_mowing_events | DNK | 2018 ===
{'dataset_id': 'EO:EEA:DAT:HRL:GRA', 'productType': 'Grassland Mowing Events', 'resolution': '10m', 'year': '2018', 'bbox': [8.076389, 54.559029, 15.193056, 57.751526], 'itemsPerPage': 200, 'startIndex': 0}
Found 24 items


✓ Saved to ../../data/LEON_P5_BII/EO_data_raw/grassland_mowing_events/grassland_mowing_events_dnk_2018

=== grassland_mowing_events | DNK | 2019 ===
{'dataset_id': 'EO:EEA:DAT:HRL:GRA', 'productType': 'Grassland Mowing Events', 'resolution': '10m', 'year': '2019', 'bbox': [8.076389, 54.559029, 15.193056, 57.751526], 'itemsPerPage': 200, 'startIndex': 0}
Found 24 items


✓ Saved to ../../data/LEON_P5_BII/EO_data_raw/grassland_mowing_events/grassland_mowing_events_dnk_2019

=== grassland_mowing_events | DNK | 2020 ===
{'dataset_id': 'EO:EEA:DAT:HRL:GRA', 'productType': 'Grassland Mowing Events', 'resolution': '10m', 'year': '2020', 'bbox': [8.076389, 54.559029, 15.193056, 57.751526], 'itemsPerPage': 200, 'startIndex': 0}
Found 24 items


✓ Saved to ../../data/LEON_P5_BII/EO_data_raw/grassland_mowing_events/grassland_mowing_events_dnk_2020

=== grassland_mowing_events | DNK | 2021 ===
{'dataset_id': 'EO:EEA:DAT:HRL:GRA', 'productType': 'Grassland Mowing Events', 'resolution': '10m', 'year': '2021', 'bbox': [8.076389, 54.559029, 15.193056, 57.751526], 'itemsPerPage': 200, 'startIndex': 0}
Found 24 items


✓ Saved to ../../data/LEON_P5_BII/EO_data_raw/grassland_mowing_events/grassland_mowing_events_dnk_2021

=== grassland_mowing_events | DNK | 2022 ===
{'dataset_id': 'EO:EEA:DAT:HRL:GRA', 'productType': 'Grassland Mowing Events', 'resolution': '10m', 'year': '2022', 'bbox': [8.076389, 54.559029, 15.193056, 57.751526], 'itemsPerPage': 200, 'startIndex': 0}
Found 24 items


...

✓ Saved to ../../data/LEON_P5_BII/EO_data_raw/grassland_mowing_events/grassland_mowing_events_nld_2019

=== grassland_mowing_events | NLD | 2020 ===
{'dataset_id': 'EO:EEA:DAT:HRL:GRA', 'productType': 'Grassland Mowing Events', 'resolution': '10m', 'year': '2020', 'bbox': [3.360782, 50.723492, 7.227095, 53.554585], 'itemsPerPage': 200, 'startIndex': 0}
Found 15 items


✓ Saved to ../../data/LEON_P5_BII/EO_data_raw/grassland_mowing_events/grassland_mowing_events_nld_2020

=== grassland_mowing_events | NLD | 2021 ===
{'dataset_id': 'EO:EEA:DAT:HRL:GRA', 'productType': 'Grassland Mowing Events', 'resolution': '10m', 'year': '2021', 'bbox': [3.360782, 50.723492, 7.227095, 53.554585], 'itemsPerPage': 200, 'startIndex': 0}
Found 15 items


✓ Saved to ../../data/LEON_P5_BII/EO_data_raw/grassland_mowing_events/grassland_mowing_events_nld_2021

=== grassland_mowing_events | NLD | 2022 ===
{'dataset_id': 'EO:EEA:DAT:HRL:GRA', 'productType': 'Grassland Mowing Events', 'resolution': '10m', 'year': '2022', 'bbox': [3.360782, 50.723492, 7.227095, 53.554585], 'itemsPerPage': 200, 'startIndex': 0}
Found 15 items


✓ Saved to ../../data/LEON_P5_BII/EO_data_raw/grassland_mowing_events/grassland_mowing_events_nld_2022

=== grassland_mowing_events | NLD | 2023 ===
{'dataset_id': 'EO:EEA:DAT:HRL:GRA', 'productType': 'Grassland Mowing Events', 'resolution': '10m', 'year': '2023', 'bbox': [3.360782, 50.723492, 7.227095, 53.554585], 'itemsPerPage': 200, 'startIndex': 0}
Found 15 items


✓ Saved to ../../data/LEON_P5_BII/EO_data_raw/grassland_mowing_events/grassland_mowing_events_nld_2023
